# 03 Gold: star schema

**Goal:** a model that analysts and BI tools can query easily.

| Table | Type | Grain |
|---|---|---|
| `gold_dim_date` | dimension | one row per month |
| `gold_dim_pollutant` | dimension | one row per pollutant |
| `gold_fact_air_quality` | fact | one row per month and pollutant that has a measurement |
| `gold_agg_yearly` | aggregate | one row per year and pollutant |

Missing months are **not** loaded into the fact table. They are reported in the data quality notebook instead.

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA air_quality")

DataFrame[]

In [0]:
%sql
-- Dimension: date. Built as a full monthly calendar between the first and last month in the data.
CREATE OR REPLACE TABLE gold_dim_date AS
WITH bounds AS (
  SELECT MIN(reading_month) AS first_month, MAX(reading_month) AS last_month
  FROM silver_air_pollution
),
calendar AS (
  SELECT EXPLODE(SEQUENCE(first_month, last_month, INTERVAL 1 MONTH)) AS full_date
  FROM bounds
)
SELECT
  YEAR(full_date) * 10000 + MONTH(full_date) * 100 + DAY(full_date)              AS date_key,
  full_date,
  YEAR(full_date)                                                                AS year,
  QUARTER(full_date)                                                             AS quarter,
  MONTH(full_date)                                                               AS month,
  CONCAT(CAST(YEAR(full_date) AS STRING), '-', LPAD(CAST(MONTH(full_date) AS STRING), 2, '0')) AS year_month
FROM calendar

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Dimension: pollutant. Units are ASSUMED (typical for air quality reporting). Confirm with the dataset source.
CREATE OR REPLACE TABLE gold_dim_pollutant AS
SELECT * FROM (VALUES
  (1, 'CO',    'Carbon monoxide',            'Gas',         'ppm'),
  (2, 'NO2',   'Nitrogen dioxide',           'Gas',         'ppm'),
  (3, 'O3',    'Ozone',                      'Gas',         'ppm'),
  (4, 'SO2',   'Sulphur dioxide',            'Gas',         'ppm'),
  (5, 'PM10',  'Particulate matter (PM10)',  'Particulate', 'ug/m3'),
  (6, 'PM2.5', 'Particulate matter (PM2.5)', 'Particulate', 'ug/m3')
) AS t(pollutant_key, pollutant_code, pollutant_name, pollutant_group, unit_assumed)

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Fact: measurements only (missing values are excluded on purpose).
CREATE OR REPLACE TABLE gold_fact_air_quality AS
SELECT
  d.date_key,
  p.pollutant_key,
  s.concentration
FROM silver_air_pollution s
JOIN gold_dim_date      d ON s.reading_month = d.full_date
JOIN gold_dim_pollutant p ON s.pollutant     = p.pollutant_code
WHERE s.is_missing = FALSE

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Aggregate: yearly average per pollutant, with year-over-year change.
-- months_with_data shows how complete each year is (2017 and 2024 are partial for some pollutants).
CREATE OR REPLACE TABLE gold_agg_yearly AS
WITH yearly AS (
  SELECT
    d.year,
    p.pollutant_code,
    COUNT(*)                          AS months_with_data,
    ROUND(AVG(f.concentration), 4)    AS avg_concentration,
    MIN(f.concentration)              AS min_concentration,
    MAX(f.concentration)              AS max_concentration
  FROM gold_fact_air_quality f
  JOIN gold_dim_date      d ON f.date_key      = d.date_key
  JOIN gold_dim_pollutant p ON f.pollutant_key = p.pollutant_key
  GROUP BY d.year, p.pollutant_code
)
SELECT
  year,
  pollutant_code,
  months_with_data,
  avg_concentration,
  min_concentration,
  max_concentration,
  ROUND(
    100.0 * (avg_concentration - LAG(avg_concentration) OVER (PARTITION BY pollutant_code ORDER BY year))
    / LAG(avg_concentration) OVER (PARTITION BY pollutant_code ORDER BY year), 1
  ) AS yoy_change_pct
FROM yearly

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_agg_yearly ORDER BY pollutant_code, year

year,pollutant_code,months_with_data,avg_concentration,min_concentration,max_concentration,yoy_change_pct
2017,CO,10,0.6459,0.53,0.716,null
2018,CO,12,0.6517,0.615,0.682,0.9
2019,CO,12,0.6501,0.565,0.888,-0.2
2020,CO,12,0.5449,0.473,0.62,-16.2
2021,CO,12,0.5338,0.488,0.564,-2.0
2022,CO,12,0.5276,0.501,0.575,-1.2
2023,CO,12,0.5551,0.522,0.621,5.2
2017,NO2,10,0.0079,0.0071,0.0087,null
2018,NO2,12,0.0077,0.0069,0.0096,-2.5
2019,NO2,12,0.0072,0.0061,0.0081,-6.5


In [0]:
%sql
SELECT * FROM gold_dim_date

date_key,full_date,year,quarter,month,year_month
20170101,2017-01-01,2017,1,1,2017-01
20170201,2017-02-01,2017,1,2,2017-02
20170301,2017-03-01,2017,1,3,2017-03
20170401,2017-04-01,2017,2,4,2017-04
20170501,2017-05-01,2017,2,5,2017-05
20170601,2017-06-01,2017,2,6,2017-06
20170701,2017-07-01,2017,3,7,2017-07
20170801,2017-08-01,2017,3,8,2017-08
20170901,2017-09-01,2017,3,9,2017-09
20171001,2017-10-01,2017,4,10,2017-10


In [0]:
%sql
SELECT * FROM gold_dim_pollutant

pollutant_key,pollutant_code,pollutant_name,pollutant_group,unit_assumed
1,CO,Carbon monoxide,Gas,ppm
2,NO2,Nitrogen dioxide,Gas,ppm
3,O3,Ozone,Gas,ppm
4,SO2,Sulphur dioxide,Gas,ppm
5,PM10,Particulate matter (PM10),Particulate,ug/m3
6,PM2.5,Particulate matter (PM2.5),Particulate,ug/m3


In [0]:
%sql
SELECT * FROM gold_fact_air_quality

date_key,pollutant_key,concentration
20170101,1,0.561
20170101,2,0.0076
20170101,3,0.015
20170101,5,38.0
20170101,4,0.0023
20170201,1,0.53
20170201,2,0.0071
20170201,3,0.018
20170201,5,40.89
20170201,4,0.0025


In [0]:
%sql
SELECT * FROM gold_agg_yearly

year,pollutant_code,months_with_data,avg_concentration,min_concentration,max_concentration,yoy_change_pct
2017,CO,10,0.6459,0.53,0.716,null
2018,CO,12,0.6517,0.615,0.682,0.9
2019,CO,12,0.6501,0.565,0.888,-0.2
2020,CO,12,0.5449,0.473,0.62,-16.2
2021,CO,12,0.5338,0.488,0.564,-2.0
2022,CO,12,0.5276,0.501,0.575,-1.2
2023,CO,12,0.5551,0.522,0.621,5.2
2017,NO2,10,0.0079,0.0071,0.0087,null
2018,NO2,12,0.0077,0.0069,0.0096,-2.5
2019,NO2,12,0.0072,0.0061,0.0081,-6.5
